In [ ]:
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv("data/sankey_data.csv")

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Filter studies to make plot clearer (studies were exluded if they were using unique tasks)
df_included = df[df['include'] == 1].copy()

# finding links between levels
domain_to_task = []
task_to_eeg = []

for _, row in df_included.iterrows():

    domains = [d.strip() for d in str(row['standardized_cognitive_domains']).split(';') if d.strip()]
    task = str(row['task_names']).strip()
    measures = [m.strip() for m in str(row['eeg_measures']).split(';') if m.strip()]
    
    for d in domains:
        domain_to_task.append((d, task))
    for m in measures:
        task_to_eeg.append((task, m))

df_d2t = pd.DataFrame(domain_to_task, columns=['source', 'target']).value_counts().reset_index(name='value')
df_t2e = pd.DataFrame(task_to_eeg, columns=['source', 'target']).value_counts().reset_index(name='value')

# sort each level
unique_domains = sorted(list(set(df_d2t['source'])))
unique_tasks = sorted(list(set(df_d2t['target']).union(set(df_t2e['source']))))
unique_eeg = sorted(list(set(df_t2e['target'])))

all_nodes = unique_domains + unique_tasks + unique_eeg
node_dict = {node: idx for idx, node in enumerate(all_nodes)}

# Preparing for the plot (color mapping and dictionaries)
domain_colors = ["#2B6CB0"] * len(unique_domains)  # Slate Blue
task_colors = ["#DD6B20"] * len(unique_tasks)      # Orange
eeg_colors = ["#285E61"] * len(unique_eeg)         # Dark Teal
node_colors = domain_colors + task_colors + eeg_colors

all_links = pd.concat([df_d2t, df_t2e], ignore_index=True)
all_links['source_id'] = all_links['source'].map(node_dict)
all_links['target_id'] = all_links['target'].map(node_dict)

# Generate the sankey plot
fig = go.Figure(data=[go.Sankey(
    arrangement="snap",
    node=dict(
        pad=18,
        thickness=18,
        line=dict(color="#1A202C", width=0.5),
        label=all_nodes,
        color=node_colors
    ),
    link=dict(
        source=all_links['source_id'],
        target=all_links['target_id'],
        value=all_links['value'],
        color="rgba(160, 174, 192, 0.35)"
    )
)])

fig.update_layout(
    font=dict(size=14, family="Inter, Arial, sans-serif"),
    height=800,
    width=1150,
    margin=dict(l=30, r=30, t=50, b=30)
)

fig.show(renderer="browser")

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Load data
df = pd.read_excel('clean_task_details.xlsx')

# Function to extract ISI numbers and convert seconds to milliseconds
def extract_isi(text):
    if not isinstance(text, str):
        return []
    
    # Check for "seconds" or "s" to identify if conversion is needed
    is_sec = bool(re.search(r'\b(seconds?|s)\b', text, re.IGNORECASE))
    
    # Find all digits/decimals
    numbers = [float(n) for n in re.findall(r'\d+(?:\.\d+)?', text)]
    
    # Convert obvious second lengths (under 20) or explicit 's' labels to ms
    result = [n * 1000 if (n < 20 or is_sec) else n for n in numbers]
    
    # Filter to standard biological/EEG task limits (100ms to 10000ms)
    return [n for n in result if 100 <= n <= 10000]

# Apply extraction and calculate the average ISI per study
df['extracted_isi'] = df['inter trial variation'].apply(extract_isi)
df['avg_isi'] = df['extracted_isi'].apply(lambda x: np.mean(x) if len(x) > 0 else np.nan)

# Set visualization style
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 1, figsize=(12, 12))
fig.suptitle("Task Timing (100ms Bins) & Trial Number Analysis", fontsize=16)

# --- Plot 1: ISI Histogram with 100ms bins ---
isi_data = df['avg_isi'].dropna()

# Create bins dynamically in 100ms steps up to the maximum ISI value
sns.histplot(
    isi_data, 
    bins=range(0, int(isi_data.max()) + 200, 100), 
    ax=axes[0], 
    color='skyblue', 
    edgecolor='black'
)

axes[0].set_title('Distribution of Average Inter-Stimulus Intervals (100ms Bins)', fontsize=14)
axes[0].set_ylabel('Number of Studies', fontsize=12)
axes[0].set_xlabel('ISI (ms)', fontsize=12)

# Set tick marks every 500ms with minor ticks at 100ms intervals
axes[0].xaxis.set_major_locator(ticker.MultipleLocator(500))
axes[0].xaxis.set_minor_locator(ticker.MultipleLocator(100))

# --- Plot 2: Box plot for nr trials per paradigm ---
# Map exact string values in the dataset to desired display labels
df['paradigm_label'] = df['paradigm'].replace({
    'CPT': 'Continuous Performance Test (CPT)',
    'Working Memory / Change Detection': 'Working Memory / Change Detection Tasks'
})

plot_order = [
    'Go_NoGo',
    'Continuous Performance Test (CPT)',
    'Stop Signal Task (SST)',
    'N-Back Task',
    'Working Memory / Change Detection Tasks'
]

# Draw the underlying box plot
sns.boxplot(
    data=df, 
    y='paradigm_label', 
    x='nr trials', 
    order=plot_order, 
    color='lightgray', 
    ax=axes[1],
    fliersize=0
)

# Overlay the individual study points (stripplot)
sns.stripplot(
    data=df, 
    y='paradigm_label', 
    x='nr trials', 
    order=plot_order, 
    alpha=0.7, 
    jitter=True, 
    size=6, 
    ax=axes[1]
)

axes[1].set_title('Number of Trials across Paradigms', fontsize=14)
axes[1].set_xlabel('Nr trials', fontsize=12)
axes[1].set_ylabel('', fontsize=12)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('isi_100ms_and_trials_plot.png', dpi=300)
plt.show()